# Notebook 05 — Cross-Source Integration JOIN

**Rubric:** §4.2 Storage (integration), §4.4 Processing

Builds `dim_unified_wallet` by joining MongoDB customer profiles with Postgres wallet aggregations. This denormalised table is the single JOIN target for all analytical queries.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
print('Working directory:', os.getcwd())

In [ ]:
from src.integration.build_unified_view import run
run()

## Verify the unified view

In [ ]:
import pandas as pd
from sqlalchemy import text
from src.config import get_pg_engine

engine = get_pg_engine()
with engine.connect() as conn:
    total = conn.execute(text('SELECT COUNT(*) FROM dim_unified_wallet')).scalar()
    sample = pd.read_sql(text('SELECT * FROM dim_unified_wallet LIMIT 5'), conn)

print(f'dim_unified_wallet: {total:,} rows')
print(sample.to_string())

In [ ]:
with engine.connect() as conn:
    coverage = pd.read_sql(text("""
        SELECT
            COUNT(*) AS total_profiles,
            SUM(CASE WHEN total_tx_count IS NOT NULL THEN 1 ELSE 0 END) AS with_tx_data,
            SUM(CASE WHEN has_xml_activity THEN 1 ELSE 0 END) AS with_xml_activity,
            COUNT(DISTINCT kyc_tier) AS distinct_kyc_tiers
        FROM dim_unified_wallet
    """), conn)
print('Coverage summary:')
print(coverage.to_string(index=False))

**Interpretation:** The unified view joins 375K customer profiles from MongoDB with transaction aggregates from Postgres. Wallets with transaction data in the Postgres sample have full enrichment; the remainder have profile data only — this is expected given the 500K-row stratified sample covers a subset of wallets.